# 04 - Business Insights
Answers to the seven HR questions, computed from the data. **Associations, not causes.**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Locate the project root whether the notebook is run from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "python"))

import config as cfg
import data_cleaning as dc
import eda_analysis as eda
import visualization as viz

pd.set_option("display.max_columns", 50)
viz.set_theme()

In [ ]:
df = eda.load_clean()
ins = eda.compute_insights(df)
T = ins['tables']
avg = ins['overall']['attrition_rate_pct']
print('Company attrition rate:', avg, '%')

## Q1. Which department has the highest attrition?

In [ ]:
d = ins['q1_top_department']
print(f"{d['Department']}: {d['AttritionRatePct']}% ({d['Attrition']} of {d['Employees']})")
display(T['by_department'])

## Q2. Which job role has the highest turnover?

In [ ]:
r = ins['q2_top_job_role']
print(f"{r['JobRole']}: {r['AttritionRatePct']}% ({r['Attrition']} of {r['Employees']})")
display(T['by_job_role'])

## Q3. Does overtime increase attrition?

In [ ]:
o = ins['q3_overtime']
print(f"Overtime {o['rate_overtime']}% vs no overtime {o['rate_no_overtime']}%  ({o['ratio']}x)")
print('Chi-square test:', o['test'])

## Q4. Does lower salary correlate with higher attrition?

In [ ]:
s = ins['q4_salary']
print(s)
display(T['by_salary_slab'])

## Q5. Which age group is most likely to leave?

In [ ]:
print(ins['q5_age']['top_group'])
display(T['by_age_group'])

## Q6. How does job satisfaction influence attrition?

In [ ]:
print(ins['q6_job_satisfaction'])
display(T['by_job_satisfaction'])

## Q7. Does promotion frequency affect retention?

In [ ]:
print('corr(years since promotion, attrition):', round(ins['q7_promotion']['corr_years_since_promotion'], 3))
print('Chi-square:', ins['q7_promotion']['test'])
display(T['by_promotion_gap'])

## Strength of association (all drivers)
Cramer's V: ~0.1 small, ~0.3 medium. No single factor is strong, so attrition is multi-factor.

In [ ]:
display(ins['chi_square_all'])

## Descriptive risk-factor count (not a predictive model)
Count of: overtime, job satisfaction <= 2, income < 3,000, tenure <= 2 years.

In [ ]:
score = ((df.OverTime == 'Yes').astype(int) + (df.JobSatisfaction <= 2).astype(int)
         + (df.MonthlyIncome < 3000).astype(int) + (df.YearsAtCompany <= 2).astype(int))
risk = df.assign(risk_factors=score).groupby('risk_factors').agg(
    employees=('AttritionFlag', 'size'), leavers=('AttritionFlag', 'sum'))
risk['attrition_rate_pct'] = (100 * risk.leavers / risk.employees).round(1)
display(risk)

## Correlation vs causation
These are snapshot associations. Leavers differ from stayers on many dimensions at once (age, pay, level, tenure), so no factor is isolated as a *cause*. Establishing causes requires exit-interview data, longitudinal tracking or controlled policy trials.

## Recommendations
See `reports/Business_Insights_Report.pdf` section *Recommendations for HR management*.